# dfguard — PySpark quickstart

This notebook demonstrates the full dfguard PySpark API end-to-end.

> **Note:** `arm()` patches all annotated functions in a module at import time and does not work inside notebooks. Use `@dfg.enforce` explicitly per function: that is the correct approach for notebook usage.

**Requirements:** `pip install 'dfguard[pyspark]'`

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

import dfguard.pyspark as dfg
from dfguard.pyspark import Optional

spark = SparkSession.builder.master("local[1]").appName("dfguard-demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 23:02:29 WARN Utils: Your hostname, Nithins-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.244 instead (on interface en1)
26/04/12 23:02:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/12 23:02:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Schema definition

Two ways to declare a schema: class-based `SparkSchema` and `schema_of()` from a live DataFrame.

Fields use **assignment form** (`=`) to avoid Pylance `reportInvalidTypeForm` warnings on PySpark DataType instances.

PySpark supports nested types natively: `T.ArrayType`, `T.StructType`, `T.MapType`. dfguard enforces them recursively.

`Optional[dtype]` marks a field as intentionally nullable. dfguard checks dtype but NOT null presence or absence.

In [2]:
# Class-based schema declaration (assignment form).
# Nested type: T.ArrayType(T.StructType([...])) — PySpark native nested type.
class OrderSchema(dfg.SparkSchema):
    order_id   = T.LongType()
    customer   = T.StringType()
    amount     = T.DoubleType()
    active     = T.BooleanType()
    # Array of structs: each order has one or more line items.
    line_items = T.ArrayType(T.StructType([
        T.StructField("sku",      T.StringType(),  True),
        T.StructField("quantity", T.IntegerType(), True),
        T.StructField("price",    T.DoubleType(),  True),
    ]))
    zip_code   = Optional[T.StringType()]   # nullable: dtype enforced, nulls allowed

print("StructType:", OrderSchema.to_struct().simpleString())

StructType: struct<order_id:bigint,customer:string,amount:double,active:boolean,line_items:array<struct<sku:string,quantity:int,price:double>>,zip_code:string>


In [3]:
# Build a matching DataFrame
rows = [
    (101, "Alice", 24.48, True,
     [{"sku": "A1", "quantity": 2, "price": 9.99},
      {"sku": "B2", "quantity": 1, "price": 4.50}],
     "94107"),
    (102, "Bob",   14.00, False,
     [{"sku": "C3", "quantity": 3, "price": 14.00}],
     None),
]
orders_df = spark.createDataFrame(rows, schema=OrderSchema.to_struct())
orders_df.printSchema()
orders_df.show(truncate=False)

root
 |-- order_id: long (nullable = false)
 |-- customer: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- active: boolean (nullable = false)
 |-- line_items: array (nullable = false)
 |    |-- element: struct (containsNull = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- zip_code: string (nullable = true)



+--------+--------+------+------+-----------------------------+--------+
|order_id|customer|amount|active|line_items                   |zip_code|
+--------+--------+------+------+-----------------------------+--------+
|101     |Alice   |24.48 |true  |[{A1, 2, 9.99}, {B2, 1, 4.5}]|94107   |
|102     |Bob     |14.0  |false |[{C3, 3, 14.0}]              |NULL    |
+--------+--------+------+------+-----------------------------+--------+



In [4]:
# schema_of() captures an exact schema from a live DataFrame.
# The resulting type requires exactly the captured columns — no extras.
CapturedSchema = dfg.schema_of(orders_df)
print("isinstance check (exact match):", isinstance(orders_df, CapturedSchema))

isinstance check (exact match): True


## 2. Schema inheritance

In [5]:
# EnrichedOrderSchema inherits all fields from OrderSchema and adds revenue.
class EnrichedOrderSchema(OrderSchema):
    revenue = T.DoubleType()

field_names = [f.name for f in EnrichedOrderSchema.to_struct().fields]
print("Enriched fields:", field_names)
assert "order_id" in field_names
assert "revenue"  in field_names
print("Inheritance confirmed.")

Enriched fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']
Inheritance confirmed.


## 3. `@dfg.enforce` decorator

Two modes:
- `@dfg.enforce` (default, `subset=True`): the DataFrame must have at least the declared columns; extras are fine.
- `@dfg.enforce(subset=False)`: exact match: the DataFrame must have only the declared columns.

In [6]:
# subset=True (default): extra columns are fine.
@dfg.enforce
def compute_revenue(df: OrderSchema):
    return df.withColumn("revenue", F.col("amount") * 1.1)

enriched_df = compute_revenue(orders_df)
print("compute_revenue passed, columns:", enriched_df.columns)

compute_revenue passed, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']


In [7]:
# subset=False: exact match — only the declared columns allowed.
@dfg.enforce(subset=False)
def write_final(df: OrderSchema) -> None:
    print(f"write_final called, columns: {df.columns}")

# PASSING: orders_df has exactly the OrderSchema columns.
write_final(orders_df)

# FAILING: enriched_df has an extra 'revenue' column.
try:
    write_final(enriched_df)
except TypeError as e:
    print(f"\nCaught expected error:\n{e}")

write_final called, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']

Caught expected error:
Schema mismatch in write_final() argument 'df':
  expected: OrderSchema
  received: order_id:bigint, customer:string, amount:double, active:boolean, line_items:array<struct<sku:string,quantity:int,price:double>>, zip_code:string, revenue:double


In [8]:
# FAILING: wrong dtype — amount as StringType instead of DoubleType.
bad_df = orders_df.withColumn("amount", F.col("amount").cast(T.StringType()))

try:
    compute_revenue(bad_df)
except TypeError as e:
    print(f"Caught expected error:\n{e}")

Caught expected error:
Schema mismatch in compute_revenue() argument 'df':
  expected: OrderSchema
  received: order_id:bigint, customer:string, amount:string, active:boolean, line_items:array<struct<sku:string,quantity:int,price:double>>, zip_code:string


## 4. `assert_valid` for load-time validation

In [9]:
# PASSING
OrderSchema.assert_valid(orders_df)
print("assert_valid passed.")

# PASSING with subset=True (default): extra columns are fine.
OrderSchema.assert_valid(enriched_df, subset=True)
print("assert_valid(subset=True) passed on DataFrame with extra column.")

# FAILING with subset=False: enriched_df has 'revenue' not in OrderSchema.
try:
    OrderSchema.assert_valid(enriched_df, subset=False)
except dfg.SchemaValidationError as e:
    print(f"\nCaught expected error:\n{e}")

assert_valid passed.
assert_valid(subset=True) passed on DataFrame with extra column.

Caught expected error:
Schema validation failed:
  ✗ Unexpected column 'revenue' (strict mode)

Schema evolution (most recent last):
  [input]  struct<order_id:bigint,customer:string,amount:double,active:boolean,line_items:array<struct<sku:string,quantity:int,price:double>>,zip_code:string,revenue:double>  (no schema change)


## 5. Schema utilities: `from_struct`, `to_code`, `diff`, `empty`

In [10]:
# from_struct: generate a SparkSchema from a live StructType.
# Nested StructTypes are recursively converted to SparkSchema subclasses.
InferredSchema = dfg.SparkSchema.from_struct(orders_df.schema, name="InferredOrderSchema")
field_names = [f.name for f in InferredSchema.to_struct().fields]
print("from_struct fields:", field_names)

from_struct fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']


In [11]:
# to_code: generate Python source for the schema class.
print(InferredSchema.to_code())

from pyspark.sql import types as T
from typing import Optional

class InferredOrderSchema(SparkSchema):
    order_id: T.LongType()
    customer: T.StringType()
    amount: T.DoubleType()
    active: T.BooleanType()
    line_items: T.ArrayType(SparkSchema, containsNull=True)
    zip_code: Optional[StringType()]


In [12]:
# diff: show differences between two schemas.
print(OrderSchema.diff(EnrichedOrderSchema))
print()
print(OrderSchema.diff(OrderSchema))

Diff OrderSchema → EnrichedOrderSchema:
  Missing column 'revenue' (expected double, nullable=False)

OrderSchema and OrderSchema are identical


In [13]:
# empty: zero-row typed dataset with the correct schema.
# Requires the SparkSession because PySpark DataFrames are always bound to a session.
empty_ds = OrderSchema.empty(spark)
print("empty() count:", empty_ds._df.count())
empty_ds._df.printSchema()

empty() count: 0
root
 |-- order_id: long (nullable = false)
 |-- customer: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- active: boolean (nullable = false)
 |-- line_items: array (nullable = false)
 |    |-- element: struct (containsNull = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- zip_code: string (nullable = true)



## 6. Schema history via `dfg.dataset()`

`dfg.dataset()` wraps a DataFrame in a tracking object. Every `withColumn`, `drop`, `select`, etc. is recorded in `schema_history`.

In [14]:
ds = dfg.dataset(orders_df)

# Each step returns a new dataset; the original is unchanged.
ds2 = ds.withColumn("revenue", F.col("amount") * 1.1)
ds3 = ds2.withColumnRenamed("customer", "customer_name")
ds4 = ds3.drop("zip_code")

# Print schema evolution.
ds4.schema_history.print()

────────────────────────────────────────────────────────────
Schema Evolution
────────────────────────────────────────────────────────────
  [ 0] input
       struct<order_id:bigint,customer:string,amount:double,active:boolean,line_items:array<struct<sku:string,quantity:int,price:double>>,zip_code:string>  (no schema change)
  [ 1] withColumn('revenue')
       added: revenue:double
  [ 2] withColumnRenamed('customer'→'customer_name')
       added: customer_name:string | dropped: customer
  [ 3] drop(['zip_code'])
       dropped: zip_code
────────────────────────────────────────────────────────────


## 7. Validate and check error messages (both passing and failing)

In [15]:
# validate() returns a list of errors without raising.
# PASSING: empty list.
errors = OrderSchema.validate(orders_df)
print("Errors on valid DataFrame:", errors)

# FAILING: missing column + wrong dtype.
broken_df = (
    orders_df
    .drop("order_id")
    .withColumn("amount", F.col("amount").cast(T.StringType()))
)
errors = OrderSchema.validate(broken_df)
print("\nErrors on broken DataFrame:")
for err in errors:
    print(" -", err)

Errors on valid DataFrame: []

Errors on broken DataFrame:
 - Missing column 'order_id' (expected bigint, nullable=False)
 - Column 'amount': type mismatch: expected double, got string


In [16]:
# FAILING: nested type mismatch — LongType instead of IntegerType in array struct.
wrong_item_schema = T.ArrayType(T.StructType([
    T.StructField("sku",      T.StringType(), True),
    T.StructField("quantity", T.LongType(),   True),   # LongType, not IntegerType
    T.StructField("price",    T.DoubleType(), True),
]))
nested_wrong_df = orders_df.withColumn(
    "line_items",
    F.col("line_items").cast(wrong_item_schema),
)
errors = OrderSchema.validate(nested_wrong_df)
print("Nested type errors:")
for err in errors:
    print(" -", err)

Nested type errors:
 - Column 'line_items[].quantity': type mismatch: expected int, got bigint


## 8. Putting it together: a typed pipeline

In [17]:
class ReportSchema(EnrichedOrderSchema):
    customer_name = T.StringType()


@dfg.enforce
def enrich(df: OrderSchema):
    step1 = df.withColumn("revenue", F.col("amount") * 1.1)
    return step1.withColumn("customer_name", F.col("customer"))


@dfg.enforce
def summarise(df: ReportSchema):
    return df.select("order_id", "customer_name", "revenue")


result = summarise(enrich(orders_df))
result.show()

+--------+-------------+------------------+
|order_id|customer_name|           revenue|
+--------+-------------+------------------+
|     101|        Alice|26.928000000000004|
|     102|          Bob|15.400000000000002|
+--------+-------------+------------------+



In [18]:
# Clean up the Spark session when done.
spark.stop()